# MedRT-SFOD — End-to-End T4 Notebook

Notebook này gom **toàn bộ bước cần/chủ yếu dùng GPU** cho project hiện tại:

1. Mount Google Drive và copy project `MedRT-SFOD` xuống `/content` để train nhanh hơn.
2. Sửa `path:` trong 2 YAML C2F trên **bản local** (không sửa file gốc trên Drive).
3. Kiểm tra đúng **YOLO26-M** (~21–22M params), tránh vô tình train scale `n`.
4. Train source detector trên **Clear Cityscapes**.
5. Stage-1 **AdaBN** trên **unlabeled Foggy Cityscapes**.
6. RASP structural audit / DepGraph go-no-go.
7. Train **Original RT-SFOD baseline**.
8. Train **RASP-SFOD**, tự resume từ state trên Drive nếu session bị ngắt.
9. Physical compact export + masked-vs-compact equivalence test.
10. Final target-label evaluation **chỉ sau training**: Source / Stage-1 / RT-SFOD / RASP compact.

> **Protocol:** Không bật `--eval` hay `--oracle_early_stop` trong Stage-1/Stage-2 main runs. Target labels chỉ được dùng ở cell final evaluation.

> **T4:** Source training + RT-SFOD baseline + RASP 60 epochs có thể dài hơn một Colab session. Notebook lưu output/checkpoint vào Drive; có thể chạy lại notebook ở session sau. RASP sẽ tự resume. Original baseline script hiện chưa có exact optimizer/teacher resume, nên baseline nên chạy trọn một lần.

## 0. Mount Drive + cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# EDIT ONLY THIS CELL FIRST
# =========================
from pathlib import Path

# Nếu folder nằm trực tiếp trong MyDrive:
DRIVE_PROJECT = Path('/content/drive/MyDrive/MedRT-SFOD')

# Bản local để code + dataset chạy nhanh trên T4
LOCAL_PROJECT = Path('/content/MedRT-SFOD')

# Output/checkpoint lưu trực tiếp vào Drive để không mất khi runtime disconnect
DRIVE_RUNS = DRIVE_PROJECT / 'runs_t4'

# Main experiment settings
DEVICE = '0'
IMGSZ = 1024
BATCH = 4              # Nếu T4 OOM: đổi thành 2
WORKERS = 2
SOURCE_EPOCHS = 100
ADABN_EPOCHS = 2
STAGE2_EPOCHS = 60
LR = 1e-4

# Save policy
BASELINE_SAVE_INTERVAL = 10
RASP_SAVE_INTERVAL = 5     # RASP resume tối đa mất ~5 epoch nếu runtime chết giữa interval

# Pipeline switches. Rerun notebook safely: completed stages are auto-skipped.
RUN_SOURCE = True
RUN_STAGE1 = True
RUN_AUDIT = True
RUN_BASELINE = True
RUN_RASP = True
RUN_EXPORT = True
RUN_FINAL_EVAL = True

# Set True only if you intentionally want to redo a completed stage.
FORCE_SOURCE = False
FORCE_STAGE1 = False
FORCE_BASELINE = False
FORCE_RASP_FROM_SCRATCH = False
FORCE_EXPORT = False

assert DRIVE_PROJECT.exists(), f'Không thấy project ở: {DRIVE_PROJECT}'
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
print('DRIVE_PROJECT =', DRIVE_PROJECT)
print('DRIVE_RUNS    =', DRIVE_RUNS)

## 1. Check T4 + copy project/dataset từ Drive xuống local

In [ ]:
import os, shutil, subprocess, sys, textwrap, json, time
import torch

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime chưa bật GPU. Colab: Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=False)

# Copy source code + dataset to local SSD. Exclude outputs/cache/env to save time/space.
if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)
LOCAL_PROJECT.mkdir(parents=True, exist_ok=True)

cmd = [
    'rsync', '-a', '--info=progress2',
    '--exclude=.git/', '--exclude=.venv/', '--exclude=__pycache__/',
    '--exclude=runs/', '--exclude=runs_t4/', '--exclude=.DS_Store',
    str(DRIVE_PROJECT) + '/', str(LOCAL_PROJECT) + '/'
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
os.chdir(LOCAL_PROJECT)
print('cwd =', Path.cwd())

## 2. Install dependencies + verify required files

In [ ]:
# Local fork is imported from current project via PYTHONPATH/cwd.
# RASP-specific dependencies:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'kneed', 'torch-pruning', 'thop'], check=True)

required = [
    'ultralytics/cfg/models/26/yolo26.yaml',
    'scripts/YOLO26/train_source_supervised.py',
    'scripts/YOLO26/stage0_stage1_adabn_rc_yolo26_v2.py',
    'scripts/YOLO26/stage2_rtsfod_yolo26.py',
    'scripts/YOLO26/rasp_pruning.py',
    'scripts/YOLO26/stage2_rasp_rtsfod_yolo26.py',
    'dataset/c2f_yolo/cityscapes/cityscapes.yaml',
    'dataset/c2f_yolo/foggy_cityscapes/foggy_cityscapes.yaml',
]
missing = [p for p in required if not (LOCAL_PROJECT/p).exists()]
if missing:
    print('MISSING FILES:')
    for p in missing: print(' -', p)
    raise FileNotFoundError('Project Drive chưa chứa đủ baseline/RASP files. Merge patch trước rồi rerun notebook.')
print('[OK] required files exist')

# Syntax check critical scripts before spending GPU hours.
for p in required[1:6]:
    if p.endswith('.py'):
        subprocess.run([sys.executable, '-m', 'py_compile', p], check=True)
print('[OK] critical scripts compile')

## 3. Make dataset YAML portable on Colab + verify dataset

In [ ]:
import yaml

SRC_YAML = LOCAL_PROJECT / 'dataset/c2f_yolo/cityscapes/cityscapes.yaml'
TGT_YAML = LOCAL_PROJECT / 'dataset/c2f_yolo/foggy_cityscapes/foggy_cityscapes.yaml'

# Only modify local copies. Drive originals stay unchanged.
def rewrite_dataset_path(yaml_path: Path, dataset_root: Path):
    d = yaml.safe_load(yaml_path.read_text())
    d['path'] = str(dataset_root.resolve())
    yaml_path.write_text(yaml.safe_dump(d, sort_keys=False), encoding='utf-8')
    return d

src_cfg = rewrite_dataset_path(SRC_YAML, LOCAL_PROJECT/'dataset/c2f_yolo/cityscapes')
tgt_cfg = rewrite_dataset_path(TGT_YAML, LOCAL_PROJECT/'dataset/c2f_yolo/foggy_cityscapes')

print('SOURCE YAML:')
print(SRC_YAML.read_text())
print('TARGET YAML:')
print(TGT_YAML.read_text())

# Count basic files; target labels can exist for final evaluation but Stage1/Stage2 do not read them for training.
def count_files(root, pattern): return len(list(Path(root).rglob(pattern)))
for name, root in [('source', Path(src_cfg['path'])), ('target', Path(tgt_cfg['path']))]:
    print(name, 'train images=', count_files(root/'images/train', '*.*'),
          'val images=', count_files(root/'images/val', '*.*'),
          'train labels=', count_files(root/'labels/train', '*.txt'),
          'val labels=', count_files(root/'labels/val', '*.txt'))

## 4. Force YOLO26-M scale and verify parameter count

In [ ]:
# Important: generic yolo26.yaml can default to first scale (n).
# Create a filename alias yolo26m.yaml so Ultralytics infers scale='m'.
BASE_CFG = LOCAL_PROJECT/'ultralytics/cfg/models/26/yolo26.yaml'
M_CFG = LOCAL_PROJECT/'ultralytics/cfg/models/26/yolo26m.yaml'
shutil.copy2(BASE_CFG, M_CFG)

# Ensure local fork is first.
sys.path.insert(0, str(LOCAL_PROJECT))
from ultralytics import YOLO

probe = YOLO(str(M_CFG), task='detect')
n_params = sum(p.numel() for p in probe.model.parameters())
print(f'YOLO26-M params = {n_params/1e6:.3f}M')
assert 15_000_000 < n_params < 30_000_000, (
    'Model không giống scale M. Dừng lại để tránh train nhầm YOLO26-N/S.'
)
del probe
torch.cuda.empty_cache()
print('[OK] YOLO26-M scale verified')

## 5. Helper paths / shell runner

In [ ]:
SOURCE_OUT = DRIVE_RUNS/'c2f_source_yolo26m'
SOURCE_BEST = SOURCE_OUT/'weights/best.pt'
SOURCE_LAST = SOURCE_OUT/'weights/last.pt'
SOURCE_DONE = SOURCE_OUT/'.complete'

STAGE1_OUT = DRIVE_RUNS/'c2f_stage1_yolo26m'
STAGE1_CKPT = STAGE1_OUT/'yolo26_stage1_adabnrc_foggy_cityscapes.pt'

AUDIT_JSON = DRIVE_RUNS/'rasp_audit_yolo26m.json'

BASELINE_OUT = DRIVE_RUNS/'c2f_rtsfod_yolo26m'
BASELINE_FINAL = BASELINE_OUT/'checkpoints'/f'yolo26_stage2_rtsfod_epoch_{STAGE2_EPOCHS}.pt'

RASP_OUT = DRIVE_RUNS/'c2f_rasp_yolo26m'
RASP_STATE_LATEST = RASP_OUT/'checkpoints/rasp_training_state_latest.pt'
RASP_STATE_FINAL = RASP_OUT/'checkpoints'/f'rasp_training_state_epoch_{STAGE2_EPOCHS}.pt'
RASP_LATENT_FINAL = RASP_OUT/'checkpoints'/f'yolo26_rasp_latent_epoch_{STAGE2_EPOCHS}.pt'
RASP_COMPACT = RASP_OUT/'yolo26m_rasp_compact.pt'

FINAL_METRICS_JSON = DRIVE_RUNS/'final_metrics_c2f.json'

for p in [SOURCE_OUT, STAGE1_OUT, BASELINE_OUT, RASP_OUT]: p.mkdir(parents=True, exist_ok=True)

def run_shell(command: str):
    print('\n>>>', command, '\n', flush=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(LOCAL_PROJECT)
    p = subprocess.run(command, shell=True, cwd=str(LOCAL_PROJECT), env=env)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {p.returncode}')

print('SOURCE_BEST   =', SOURCE_BEST)
print('STAGE1_CKPT   =', STAGE1_CKPT)
print('BASELINE_FINAL=', BASELINE_FINAL)
print('RASP_OUT      =', RASP_OUT)

## 6. GPU Stage -1 — train source YOLO26-M on Clear Cityscapes

In [ ]:
if RUN_SOURCE:
    if SOURCE_DONE.exists() and SOURCE_BEST.exists() and not FORCE_SOURCE:
        print('[SKIP] source training already marked complete:', SOURCE_BEST)
    else:
        if FORCE_SOURCE and SOURCE_OUT.exists():
            print('[FORCE] removing previous source output')
            shutil.rmtree(SOURCE_OUT)
            SOURCE_OUT.mkdir(parents=True, exist_ok=True)

        # If an interrupted Ultralytics run has last.pt, exact resume from that checkpoint.
        if SOURCE_LAST.exists() and not FORCE_SOURCE:
            print('[RESUME] source training from', SOURCE_LAST)
            model = YOLO(str(SOURCE_LAST))
            model.train(resume=True)
            del model
        else:
            run_shell(
                f'python scripts/YOLO26/train_source_supervised.py '
                f'--model-cfg "{M_CFG}" '
                f'--data "{SRC_YAML}" --task detect '
                f'--epochs {SOURCE_EPOCHS} --imgsz {IMGSZ} --batch {BATCH} '
                f'--device {DEVICE} --out-dir "{SOURCE_OUT}"'
            )
        assert SOURCE_BEST.exists(), f'Missing source best.pt: {SOURCE_BEST}'
        SOURCE_DONE.write_text('complete\n')
        print('[OK] source best =', SOURCE_BEST)
else:
    print('[SKIP by switch] RUN_SOURCE=False')

## 7. GPU Stage 1 — AdaBN on unlabeled Foggy Cityscapes

In [ ]:
if RUN_STAGE1:
    assert SOURCE_BEST.exists(), 'Need source best.pt first.'
    if STAGE1_CKPT.exists() and not FORCE_STAGE1:
        print('[SKIP] Stage1 exists:', STAGE1_CKPT)
    else:
        run_shell(
            f'python scripts/YOLO26/stage0_stage1_adabn_rc_yolo26_v2.py '
            f'--weights "{SOURCE_BEST}" --data "{TGT_YAML}" '
            f'--out_dir "{STAGE1_OUT}" --imgsz {IMGSZ} --batch {BATCH} '
            f'--workers {WORKERS} --epochs_adabn {ADABN_EPOCHS} --epochs_rc 0 '
            f'--device {DEVICE}'
        )
        assert STAGE1_CKPT.exists(), f'Stage1 checkpoint not found: {STAGE1_CKPT}'
    print('[OK] Stage1 =', STAGE1_CKPT)
else:
    print('[SKIP by switch] RUN_STAGE1=False')

## 8. GPU RASP audit — go/no-go before 60-epoch training

Cell này không phụ thuộc `inspect_rasp_prunable_groups.py`; nó audit trực tiếp bằng `rasp_pruning.py` để notebook self-contained hơn.

- Discover standard YOLO Bottleneck hidden groups.
- Run one dummy forward to observe feature sizes and estimate per-hidden-channel MAC cost.
- Run Torch-Pruning DepGraph locality audit.
- Save JSON to Drive.

**Không dùng target labels.**

In [ ]:
if RUN_AUDIT:
    assert STAGE1_CKPT.exists(), 'Need Stage1 checkpoint first.'
    from scripts.YOLO26.rasp_pruning import AdaptiveHiddenPruner, save_audit_json

    audit_wrapper = YOLO(str(STAGE1_CKPT))
    audit_model = audit_wrapper.model.to('cuda:0').float().eval()
    pruner = AdaptiveHiddenPruner(audit_model)
    assert pruner.groups, 'RASP found 0 eligible Bottleneck groups.'

    # Observe H/W for MAC estimates with a small dummy image.
    pruner.install()
    dummy = torch.zeros(1, 3, 256, 256, device='cuda:0')
    with torch.no_grad():
        _ = audit_model(dummy)
    pruner.uninstall()

    total_prunable_macs, total_prunable_params = pruner.total_prunable_cost()
    total_params = sum(p.numel() for p in audit_model.parameters())
    total_hidden = sum(g.hidden for g in pruner.groups.values())

    dep = pruner.audit_depgraph(dummy, require_local=True)
    dep_ok = sum(int(v['ok']) for v in dep.values())

    audit_summary = {
        'model': str(STAGE1_CKPT),
        'eligible_hidden_groups': len(pruner.groups),
        'eligible_hidden_channels': int(total_hidden),
        'model_total_params': int(total_params),
        'controlled_params_est': float(total_prunable_params),
        'controlled_params_fraction_est': float(total_prunable_params / max(total_params, 1)),
        'controlled_macs_est_256': float(total_prunable_macs),
        'depgraph_local_safe': int(dep_ok),
        'depgraph_total': len(dep),
        'depgraph': dep,
    }
    AUDIT_JSON.write_text(json.dumps(audit_summary, indent=2), encoding='utf-8')
    print(json.dumps({k:v for k,v in audit_summary.items() if k!='depgraph'}, indent=2))
    print('audit saved ->', AUDIT_JSON)

    if dep_ok != len(dep):
        print('WARNING: Some groups are not DepGraph-local-safe. Do NOT enable --rasp_require_depgraph until reviewed.')
    if audit_summary['controlled_params_fraction_est'] < 0.10:
        print('WARNING: controllable parameter space <10% of model. Consider expanding pruning space before full study.')

    del audit_model, audit_wrapper, pruner, dummy
    torch.cuda.empty_cache()
else:
    print('[SKIP by switch] RUN_AUDIT=False')

## 9. GPU Original RT-SFOD baseline — 60 epochs

Giữ `--eval` **OFF**. Final target evaluation nằm ở cuối notebook.

Nếu baseline đã hoàn tất (`epoch_60.pt`) notebook tự skip. Baseline script gốc chưa lưu optimizer+teacher state để exact resume; nếu runtime chết giữa baseline, cần chạy lại baseline từ Stage-1.

In [ ]:
if RUN_BASELINE:
    assert STAGE1_CKPT.exists(), 'Need Stage1 checkpoint first.'
    if BASELINE_FINAL.exists() and not FORCE_BASELINE:
        print('[SKIP] baseline final exists:', BASELINE_FINAL)
    else:
        if FORCE_BASELINE and BASELINE_OUT.exists():
            shutil.rmtree(BASELINE_OUT)
            BASELINE_OUT.mkdir(parents=True, exist_ok=True)
        run_shell(
            f'python scripts/YOLO26/stage2_rtsfod_yolo26.py '
            f'--stage1_model "{STAGE1_CKPT}" --data "{TGT_YAML}" '
            f'--out_dir "{BASELINE_OUT}" --imgsz {IMGSZ} --batch {BATCH} '
            f'--workers {WORKERS} --epochs {STAGE2_EPOCHS} --lr {LR} '
            f'--device {DEVICE} --save_interval {BASELINE_SAVE_INTERVAL}'
        )
        assert BASELINE_FINAL.exists(), f'Missing baseline final: {BASELINE_FINAL}'
    print('[OK] baseline final =', BASELINE_FINAL)
else:
    print('[SKIP by switch] RUN_BASELINE=False')

## 10. GPU RASP-SFOD — adaptive in-loop pruning

Defaults giữ đúng method đã chốt:

- Dense Teacher + same YOLO26-M Student
- target Taylor EMA
- per-block GMM redundancy discovery
- cost-aware global ranking
- Kneedle/max-distance adaptive budget
- DHF reliability gate
- 5-epoch warmup, 3-epoch recovery interval
- 8-channel packs
- max 5% prunable MAC cost per pruning event

Notebook tự dùng `rasp_training_state_latest.pt` để resume nếu có.

In [ ]:
if RUN_RASP:
    assert STAGE1_CKPT.exists(), 'Need Stage1 checkpoint first.'

    if RASP_STATE_FINAL.exists() and RASP_LATENT_FINAL.exists() and not FORCE_RASP_FROM_SCRATCH:
        print('[SKIP] RASP final exists:', RASP_STATE_FINAL)
    else:
        if FORCE_RASP_FROM_SCRATCH and RASP_OUT.exists():
            print('[FORCE] removing previous RASP run')
            shutil.rmtree(RASP_OUT)
            RASP_OUT.mkdir(parents=True, exist_ok=True)

        resume_arg = ''
        if RASP_STATE_LATEST.exists() and not FORCE_RASP_FROM_SCRATCH:
            resume_arg = f' --resume_state "{RASP_STATE_LATEST}"'
            print('[RESUME] RASP from:', RASP_STATE_LATEST)

        run_shell(
            f'python scripts/YOLO26/stage2_rasp_rtsfod_yolo26.py '
            f'--stage1_model "{STAGE1_CKPT}" --data "{TGT_YAML}" '
            f'--out_dir "{RASP_OUT}" --imgsz {IMGSZ} --batch {BATCH} '
            f'--workers {WORKERS} --epochs {STAGE2_EPOCHS} --lr {LR} '
            f'--device {DEVICE} --save_interval {RASP_SAVE_INTERVAL} '
            f'--rasp_enable --rasp_warmup_epochs 5 --rasp_cycle_epochs 3 '
            f'--rasp_reliability_threshold 0.50 --rasp_importance_beta 0.90 '
            f'--rasp_min_hidden 16 --rasp_min_keep_ratio 0.50 --rasp_round_to 8 '
            f'--rasp_gmm_posterior 0.80 --rasp_gmm_bic_gain 10 '
            f'--rasp_gmm_min_separation 1.0 --rasp_gmm_min_samples 16 '
            f'--rasp_cost_gamma 1.0 --rasp_max_step_cost_fraction 0.05 '
            f'{resume_arg}'
        )

        assert RASP_STATE_FINAL.exists(), f'Missing RASP final state: {RASP_STATE_FINAL}'
        assert RASP_LATENT_FINAL.exists(), f'Missing RASP latent final: {RASP_LATENT_FINAL}'
    print('[OK] RASP final state =', RASP_STATE_FINAL)
else:
    print('[SKIP by switch] RUN_RASP=False')

## 11. GPU physical compact export + numerical equivalence

Training-time masks **không** làm giảm stored parameter count. Cell này mới thật sự remove hidden channels khỏi `cv1.out` / `cv2.in` và tạo compact checkpoint.

Verification bắt buộc:

`masked dense-latent Student ≈ physical compact Student`

In [ ]:
if RUN_EXPORT:
    assert RASP_STATE_FINAL.exists() and RASP_LATENT_FINAL.exists(), 'Need final RASP run first.'
    if RASP_COMPACT.exists() and not FORCE_EXPORT:
        print('[SKIP] compact model exists:', RASP_COMPACT)
    else:
        from scripts.YOLO26.rasp_pruning import (
            export_compact_model, pruner_from_state, recursive_max_abs_diff
        )

        state = torch.load(RASP_STATE_FINAL, map_location='cpu', weights_only=False)
        rasp_state = state['rasp']
        assert rasp_state.get('enabled', False), 'Final state does not contain enabled RASP.'

        # A) masked dense-latent model
        masked_wrap = YOLO(str(RASP_LATENT_FINAL))
        masked_model = masked_wrap.model.to('cuda:0').float().eval()
        masked_pruner = pruner_from_state(masked_model, rasp_state['pruner'])
        masked_pruner.load_state_dict(rasp_state['pruner'], strict=True)
        masked_pruner.install()

        # B) physically compact model from a separate clean latent copy
        compact_wrap = YOLO(str(RASP_LATENT_FINAL))
        latent_for_compact = compact_wrap.model.to('cuda:0').float().eval()
        compact_model = export_compact_model(latent_for_compact, rasp_state).to('cuda:0').eval()

        x = torch.randn(1, 3, 256, 256, device='cuda:0')
        with torch.no_grad():
            y_masked = masked_model(x)
            y_compact = compact_model(x)
        max_diff = recursive_max_abs_diff(y_masked, y_compact)
        print('masked_vs_compact_max_abs_diff =', max_diff)
        assert max_diff < 1e-4, f'Physical export equivalence failed: diff={max_diff}'

        dense_params = sum(p.numel() for p in masked_model.parameters())
        compact_params = sum(p.numel() for p in compact_model.parameters())
        print(f'dense latent params = {dense_params/1e6:.3f}M')
        print(f'compact params      = {compact_params/1e6:.3f}M')
        print(f'param reduction     = {(1-compact_params/dense_params)*100:.2f}%')

        # Save as standard Ultralytics checkpoint.
        compact_wrap.model = compact_model
        compact_wrap.save(str(RASP_COMPACT))
        print('saved compact ->', RASP_COMPACT)

        masked_pruner.uninstall()
        del masked_model, masked_wrap, compact_model, compact_wrap, latent_for_compact, x
        torch.cuda.empty_cache()
else:
    print('[SKIP by switch] RUN_EXPORT=False')

## 12. GPU final evaluation — target labels allowed **only now**

Đây là final reporting, không phải model selection. Evaluates:

- Source-only YOLO26-M
- Stage-1 AdaBN
- Original RT-SFOD
- RASP compact Student

Saves one JSON to Drive.

In [ ]:
if RUN_FINAL_EVAL:
    candidates = {
        'source_only': SOURCE_BEST,
        'stage1_adabn': STAGE1_CKPT,
        'rtsfod_baseline': BASELINE_FINAL,
        'rasp_compact': RASP_COMPACT,
    }
    results = {}
    for name, path in candidates.items():
        if not Path(path).exists():
            print('[SKIP missing]', name, path)
            continue
        print('\n=== FINAL EVAL:', name, '===')
        m = YOLO(str(path))
        metrics = m.val(
            data=str(TGT_YAML), imgsz=IMGSZ, batch=BATCH,
            device=DEVICE, conf=0.001, iou=0.6,
            plots=False, verbose=False,
        )
        box = metrics.box
        params = sum(p.numel() for p in m.model.parameters())
        results[name] = {
            'checkpoint': str(path),
            'params': int(params),
            'params_M': float(params/1e6),
            'mAP50_95': float(box.map),
            'mAP50': float(box.map50),
            'mAP75': float(box.map75),
            'precision': float(box.mp),
            'recall': float(box.mr),
        }
        print(json.dumps(results[name], indent=2))
        del m
        torch.cuda.empty_cache()

    FINAL_METRICS_JSON.write_text(json.dumps(results, indent=2), encoding='utf-8')
    print('\nSaved final metrics ->', FINAL_METRICS_JSON)
else:
    print('[SKIP by switch] RUN_FINAL_EVAL=False')

## 13. Inspect RASP pruning history

In [ ]:
history_file = RASP_OUT/'rasp_history.jsonl'
if history_file.exists():
    import pandas as pd
    rows = [json.loads(x) for x in history_file.read_text().splitlines() if x.strip()]
    df = pd.DataFrame(rows)
    cols = [c for c in [
        'epoch','loss','det','mard','avg_dhf_conf',
        'rasp_status','rasp_hidden_sparsity','rasp_saved_prunable_macs_frac',
        'rasp_saved_prunable_params_frac','rasp_new_packs','rasp_candidates',
        'rasp_knee_packs','rasp_knee_method'
    ] if c in df.columns]
    display(df[cols].tail(30))
else:
    print('No RASP history yet:', history_file)

## Session restart guide

Mỗi lần Colab mở session mới:

1. Bật **T4 GPU**.
2. Mở notebook này từ Drive.
3. Chạy từ đầu.
4. Notebook copy project/dataset từ Drive xuống `/content` lại.
5. Các stage đã complete sẽ tự skip.
6. Nếu RASP chưa xong nhưng có `rasp_training_state_latest.pt`, RASP cell tự resume.

Không cần đổi YAML trên Mac/Drive: notebook luôn rewrite **local YAML only**.

Nếu T4 OOM ở `1024 / batch=4`, đổi `BATCH=2` trong config cell. Không thay `IMGSZ` cho main comparison nếu bạn muốn giữ protocol nhất quán.